Parallel Workflow

In [ ]:
# !pip install langchain langgraph dotenv

In [ ]:
from langgraph.graph import StateGraph
from typing import TypedDict, List
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

In [ ]:
# Create a OpenAI model instance
from langchain_openai import ChatOpenAI

# model = ChatOpenAI()


model = ChatOpenAI(    
    base_url="http://localhost:12434/engines/v1",
    api_key="docker", 
    temperature=0, 
    model = "ai/smollm2:360M-Q4_K_M")

In [ ]:
# Define agent's state

class AgentState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    strike_rate: float
    boundary_percentage: float
    balls_per_boundary: float
    summary: str

In [ ]:
def calculate_strike_rate_helper(runs: int, balls: int) -> float:
    if balls == 0:
        return 0.0
    return (runs / balls) * 100

def calculate_boundary_percentage_helper(fours: int, sixes: int, balls: int) -> float:
    if balls == 0:
        return 0.0
    boundaries = fours + sixes
    return (boundaries / balls) * 100

def calculate_balls_per_boundary_helper(fours: int, sixes: int, balls: int) -> float:
    boundaries = fours + sixes
    if boundaries == 0:
        return float('inf')  # Return infinity if no boundaries were hit
    return balls / boundaries

In [ ]:
def calculate_strike_rate(state: AgentState):
    return {'strike_rate': calculate_strike_rate_helper(state['runs'], state['balls'])}

def calculate_boundary_percentage(state: AgentState):
    return {'boundary_percentage': calculate_boundary_percentage_helper(state['fours'], state['sixes'], state['balls'])}

def calculate_balls_per_boundary(state: AgentState):
    return {'balls_per_boundary': calculate_balls_per_boundary_helper(state['fours'], state['sixes'], state['balls'])}

def summary(state: AgentState) -> AgentState:
    """Create content for the article based on the outline."""
    # Placeholder implementation - replace with actual content creation logic
    # return {"content": f"Content based on {state['outline']}"}
    
    # Fetch title and outline from state
    information = f"Runs: {state['runs']}, Balls: {state['balls']}, Fours: {state['fours']}, Sixes: {state['sixes']}, Strike Rate: {state['strike_rate']:.2f}, Boundary Percentage: {state['boundary_percentage']:.2f}, Balls per Boundary: {state['balls_per_boundary']:.2f}"

    # Generate content based on outline using LLM (placeholder logic)
    content = f"Generate summary for the batsman based on this information: {information}"
    content = model.invoke(content).content

    state["summary"] = content
    return state


In [ ]:
# Create grpah
from langgraph.graph import START, END


# Create a workflow graph
graph = StateGraph(AgentState)

# add nodes
graph.add_node("calculate_strike_rate", calculate_strike_rate)
graph.add_node("calculate_boundary_percentage", calculate_boundary_percentage)
graph.add_node("calculate_balls_per_boundary", calculate_balls_per_boundary)
graph.add_node("summary", summary)

# add edges
graph.add_edge(START, "calculate_strike_rate")
graph.add_edge(START, "calculate_boundary_percentage")
graph.add_edge(START, "calculate_balls_per_boundary")
graph.add_edge("calculate_strike_rate", "summary")
graph.add_edge("calculate_boundary_percentage", "summary")
graph.add_edge("calculate_balls_per_boundary", "summary")
graph.add_edge("summary", END)

# compile graph
workflow = graph.compile() 

In [ ]:
# visualize graph
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())

In [ ]:
# execute graph
initial_state = AgentState({
    "runs": 150,
    "balls": 100,
    "fours": 15,
    "sixes": 5,
    "strike_rate": 1.5,
    "boundary_percentage": 20.0,
    "balls_per_boundary": 5.0,
    "summary": ""
})

final_state = workflow.invoke(initial_state)
print("summary:", final_state["summary"])